## 2. Linear Regression Baseline

In [3]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [4]:
import json

from src import dataloader as data
from src import metrics
from src.models.classical_baseline import LinearRegressionPredictor, HORIZONS

MODEL_NAME = 'LinearRegression'
# Built from REPO_ROOT, not the current working directory: Colab's cwd is /content regardless of
# where this notebook file itself lives on the mounted Drive, so a plain relative path like
# Path('results') would silently write to the wrong (ephemeral) place.
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_PATH = EXPERIMENT_DIR / 'results' / 'lr_results.json'

In [5]:
stats = data.load_cgm_stats()
X_train, y_train = data.load_arrays('train')
X_val, y_val = data.load_arrays('val')
X_test, y_test = data.load_arrays('test')

print('train', X_train.shape, 'val', X_val.shape, 'test', X_test.shape)

train (83532, 24) val (16739, 24) test (16842, 24)


In [6]:
model = LinearRegressionPredictor().fit(X_train, y_train)

In [7]:
val_results = metrics.evaluate(model.predict(X_val), y_val, stats['cgm_mean'], stats['cgm_std'])
test_results = metrics.evaluate(model.predict(X_test), y_test, stats['cgm_mean'], stats['cgm_std'])

print('=== Validation ===')
print(metrics.format_results_table(val_results))
print()
print('=== Test ===')
print(metrics.format_results_table(test_results))

=== Validation ===
Horizon (min) |     RMSE |      MAE |    gRMSE | Zone A+B %
-----------------------------------------------------------
           15 |    11.43 |     7.74 |    12.40 |      99.78
           30 |    18.39 |    12.42 |    20.73 |      99.38
           60 |    25.86 |    17.42 |    30.66 |      98.72
           90 |    30.25 |    20.53 |    36.63 |      98.17
          120 |    32.91 |    22.62 |    40.26 |      97.81

=== Test ===
Horizon (min) |     RMSE |      MAE |    gRMSE | Zone A+B %
-----------------------------------------------------------
           15 |    11.57 |     7.72 |    12.82 |      99.86
           30 |    19.25 |    12.63 |    22.43 |      99.61
           60 |    28.14 |    18.26 |    34.55 |      98.85
           90 |    33.62 |    21.90 |    42.16 |      98.05
          120 |    36.96 |    24.21 |    46.79 |      97.33


In [8]:
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seed': None,  # OLS closed-form solution -- deterministic, no seed applies
        'val_results': val_results,
        'test_results': test_results,
    }, f, indent=2)

print(f'saved to {RESULTS_PATH}')

saved to /Users/ucabtt4/Projects/Dissertation/dissertation/experiments/1_baseline/results/lr_results.json
